In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:

-- RDC_WORKS 2 STEP


-- STEP 1

CREATE or replace TABLE ADC_WORKS_ARTISTS_COMPOSERS_MATCHED AS (
SELECT DISTINCT
    t1.RDC_WORKS_ID,
    t1.APRA_WORK_ID,
    t1.APRA_ORIGINAL_TITLE as APRA_TITLE,
    t1.APRA_ISWC,
    t1.MUZOOKA_TRACK_ID,
    UPPER(t1.MUZOOKA_ORIGINAL_TITLE) as MUZOOKA_TITLE,
    --t1.MUZOOKA_CLEANED_TITLE as MUZOOKA_TITLE,
    t1.MUZOOKA_ISWC,
    (t1.MATCH_SCORE)/100 AS TITLE_MATCH_SCORE,
    t1.YN_ISWC_MATCH,
    t3.adc_total_composers AS APRA_COMPOSER_CT,
    t3.mzk_total_composers AS MUZOOKA_COMPOSER_CT,
    (t3.composer_mtch_pcntg)*100 as WRITER_MATCH_PERCENTAGE,
    t4.artist_match_total AS ARTIST_MATCH_SCORE,
    t1.CD_TYPE,
    CASE 
        WHEN t1.YN_PERF_OWNERSHIP = 'N' 
             AND (LENGTH(t2.COMPOSER_NAMES) = 40 
                  OR (LENGTH(t2.COMPOSER_NAMES) = 39 AND RIGHT(t2.COMPOSER_NAMES, 1) = ' ')) 
                      AND (t3.adc_total_composers < t3.mzk_total_composers)
        THEN 'Y'
        ELSE 'N'
    END AS YN_COMPOSERS_TRUNCATED
FROM ADC_WORKS_COLUMNS_TITLE_MATCHES_COMBINED t1
INNER JOIN ADCWORKS t2 
            ON t1.APRA_WORK_ID = t2.APRA_WORK_ID 
INNER JOIN COMPOSER_PART_BY_PART_MATCH_FULL t3 
            ON t1.APRA_WORK_ID = t3.APRA_WORK_ID 
            AND T1.MUZOOKA_TRACK_ID = T3.MUZOOKA_TRACK_ID
INNER JOIN ARTIST_PART_BY_PART_MATCH_FULL t4
            ON t1.APRA_WORK_ID = t4.APRA_WORK_ID 
            AND T1.MUZOOKA_TRACK_ID = T4.MUZOOKA_TRACK_ID
--WHERE T1.APRA_WORK_ID = 'GW33624310'
);

-- STEP 2

INSERT INTO EDW_APPS.MATCHING.RDC_WORKS (
    RDC_WORKS_ID,
    APRA_WORK_ID,
    APRA_TITLE,
    MUZOOKA_TRACK_ID,
    MUZOOKA_TITLE,
    TITLE_MATCH_SCORE,
    APRA_ISWC,
    MUZOOKA_ISWC,
    YN_MATCH_ISWC,
    APRA_COMPOSER_COUNT,
    MUZOOKA_COMPOSER_COUNT,
    WRITER_MATCH_PCNT,
    ARTIST_MATCH_SCORE,
    YN_COMPOSERS_TRUNCATED
)
SELECT DISTINCT
    RDC_WORKS_ID,
    APRA_WORK_ID,
    APRA_TITLE,
    MUZOOKA_TRACK_ID,
    MUZOOKA_TITLE,
    TITLE_MATCH_SCORE,
    APRA_ISWC,
    MUZOOKA_ISWC,
    YN_ISWC_MATCH AS YN_MATCH_ISWC,
    APRA_COMPOSER_CT AS APRA_COMPOSER_COUNT,
    MUZOOKA_COMPOSER_CT AS MUZOOKA_COMPOSER_COUNT,
    WRITER_MATCH_PERCENTAGE AS WRITER_MATCH_PCNT,
    ARTIST_MATCH_SCORE,
    YN_COMPOSERS_TRUNCATED
FROM ADC_WORKS_ARTISTS_COMPOSERS_MATCHED;






In [ ]:
--CREATE RDC COMPOSERS

INSERT INTO EDW_APPS.MATCHING.RDC_COMPOSERS (
    RDC_WORKS_ID,
    APRA_WORK_ID,
    MUZOOKA_TRACK_ID,
    APRA_IPI,
    YN_MATCH_IPI,
    APRA_NAME,
    NAME_MATCH_SCORE
)
SELECT DISTINCT 
    c.RDC_WORKS_ID,
    c.APRA_WORK_ID,
    c.MUZOOKA_TRACK_ID,
    c.ADC_IPI AS APRA_IPI,
    CASE 
        WHEN c.ADC_IPI = comp.ipi THEN 'Y' 
        ELSE 'N' 
    END AS YN_MATCH_IPI,
    c.ADC_COMPOSER_NAME AS APRA_NAME,
    c.NAME_MATCH_SCORE
FROM COMPOSER_PART_BY_PART_MATCH_FULL c
INNER JOIN composers comp ON c.MUZOOKA_TRACK_ID = UPPER(comp.track_id);

In [ ]:
-- CREATE RDC ARTISTS

INSERT INTO EDW_APPS.MATCHING.RDC_ARTISTS (
    RDC_WORKS_ID,
    APRA_WORK_ID,
    MUZOOKA_TRACK_ID,
    APRA_ARTIST_ID,
    APRA_NAME,
    APRA_ARTIST_CT,
    ARTIST_MATCH_SCORE
)
SELECT DISTINCT
    RDC_WORKS_ID,
    APRA_WORK_ID,
    MUZOOKA_TRACK_ID,
    ADC_ARTIST_ID AS APRA_ARTIST_ID,
    ADC_ARTIST_NAME AS APRA_NAME,
    ADC_ARTIST_CT AS APRA_ARTIST_CT,
    ARTIST_MATCH_TOTAL AS ARTIST_MATCH_SCORE
FROM ARTIST_PART_BY_PART_MATCH_FULL;




In [ ]:
-- CREATE RDC ISRC


--STEP 1 

CREATE OR REPLACE TABLE ISRC_WITH_WORKS_TRACKS AS (
WITH DISTINCT_TITLE_MATCHES AS (
    SELECT DISTINCT 
           APRA_WORK_ID,
           MUZOOKA_TRACK_ID,
           RDC_WORKS_ID
    FROM ADC_WORKS_COLUMNS_TITLE_MATCHES_COMBINED_1
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY i.ADC_ISRC_ID) AS RDC_ISRC_ID,
    c.RDC_WORKS_ID,
    i.APRA_WORK_ID,
    c.MUZOOKA_TRACK_ID,
    i.ISRC,
    CASE WHEN r.isrc = i.ISRC THEN 'Y' ELSE 'N' END AS YN_ISRC_MATCH,
FROM 
    DISTINCT_TITLE_MATCHES c
    INNER JOIN ADCISRC i
        ON i.APRA_WORK_ID = c.APRA_WORK_ID
    INNER JOIN recordings r
        ON c.MUZOOKA_TRACK_ID = UPPER(r.track_id)
--WHERE I.APRA_WORK_ID = 'GW33624310'
);





-- STEP 2 

-- Improved to handle leading/trailing spaces + hyphens 
-- SNOWFLAKE WOULDVE TRUNCATED ISRCs WITH LEN > 12 OR LEADING/TRAILING SPACES

INSERT INTO EDW_APPS.MATCHING.RDC_ISRC (
    RDC_WORKS_ID,
    APRA_WORK_ID,
    MUZOOKA_TRACK_ID,
    ISRC,
    YN_MATCH_ISRC
)
SELECT 
    RDC_WORKS_ID,
    APRA_WORK_ID,
    MUZOOKA_TRACK_ID,
    LEFT(REPLACE(REPLACE(TRIM(ISRC), '-', ''), ' ', ''), 12),
    YN_ISRC_MATCH AS YN_MATCH_ISRC
FROM ISRC_WITH_WORKS_TRACKS;

